# Bajaj — warranty-km projection & cycle-based RUL

Charge cycles are **not** a health signal (see `../05_insights/cycle_features_rejected.ipynb`), but they're fine for the jobs they *are* suited to: projecting when a vehicle hits its **km warranty limit**, and expressing **remaining life in cycles**.

Bajaj is the right OEM for this — high-mileage cargo 3-wheelers with a **real BMS cycle counter**. Unlike Mahindra (where the km limit never binds), here **km is the binding constraint**.

In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import config
bj = pd.read_parquet('data/bajaj/features/feature_table.parquet').sort_values(['vin','age_months'])
WYRS, WKM = config.warranty_for('bajaj', '')
print(f'Bajaj warranty: {WYRS} yr / {WKM:,} km (whichever first) | {bj.vin.nunique()} vehicles')

## Per-vehicle usage rates

Recent km/month, cycles/month, current odometer & cycle count, and the reported-SoH fade rate (pp/month). EoL is set to **70%** because these aged cargo units are already below the 80% first-life threshold.

In [ ]:
EOL = 70.0
rows=[]
for v,g in bj.groupby('vin'):
    g=g.reset_index(drop=True)
    span=max(g['age_months'].iloc[-1]-g['age_months'].iloc[0], 1e-9)
    odo_now=g['odo_max'].iloc[-1]; cyc_now=g['cyc_max'].iloc[-1]; soh_now=g['soh'].iloc[-1]
    kmpm=(odo_now-g['odo_max'].iloc[0])/span; cpm=(cyc_now-g['cyc_max'].iloc[0])/span
    sslope=(g['soh'].iloc[0]-soh_now)/span
    rows.append(dict(vin=v[-8:], odo_km=int(odo_now), km_per_mo=round(kmpm), cycles=int(cyc_now),
                     cyc_per_mo=round(cpm,1), soh=round(soh_now), pp_per_mo=round(sslope,2),
                     km_per_cycle=round(kmpm/cpm,1) if cpm>0 else np.nan))
veh=pd.DataFrame(rows); veh

## Warranty-km projection

Months until each vehicle reaches the 120,000 km warranty limit at its recent km/month.

In [ ]:
veh['km_warranty_mo']=np.where(veh.odo_km>=WKM, np.inf,
                                np.where(veh.km_per_mo>0,(WKM-veh.odo_km)/veh.km_per_mo, np.inf))
n_past=(veh.odo_km>=WKM).sum()
print(f'Already PAST {WKM:,} km: {n_past}/{len(veh)} | km/month range {veh.km_per_mo.min():.0f}-{veh.km_per_mo.max():.0f}')
o=veh.sort_values('km_warranty_mo')
col=['tab:red' if p else 'tab:orange' for p in (o.odo_km>=WKM)]
plt.figure(figsize=(9,4.2))
plt.barh(o.vin, np.where(np.isfinite(o.km_warranty_mo), o.km_warranty_mo, 0), color=col)
plt.xlabel('months until 120,000 km limit'); plt.title('Bajaj — km-warranty runway (red = already past)')
plt.tight_layout()

**Finding:** for Bajaj the **km limit binds** — vehicles exhaust the 120k km warranty in months, and some already have. This is the opposite of the Mahindra fleet, where low mileage means the *time/SoH* limit binds and km never does (playbook §7 insight 5).

## Cycle-based RUL

Cycles aren't a health signal, but they're a usage *clock*: combine the reported-SoH fade rate with the cycle rate to express remaining life to EoL in **both months and cycles**.

In [ ]:
veh['mo_to_eol']=np.where(veh.pp_per_mo>0,(veh.soh-EOL)/veh.pp_per_mo, np.inf)
veh['cycles_to_eol']=np.where(np.isfinite(veh.mo_to_eol), (veh.mo_to_eol*veh.cyc_per_mo).round(), np.nan)
print(f'From now to {EOL:.0f}% SoH (median): {np.nanmedian(veh.mo_to_eol):.0f} months  ~ '
      f'{np.nanmedian(veh.cycles_to_eol):.0f} more cycles, at current usage')
veh[['vin','odo_km','soh','pp_per_mo','cyc_per_mo','km_warranty_mo','mo_to_eol','cycles_to_eol']].round(1)

### Caveats
- **No registration date** in the Bajaj telemetry, so the *time* warranty (3 yr) can't be anchored precisely —   age is measured from first observed month. The **km** projection is solid; the time projection is not.
- **Rated cycle life is unknown** (no Bajaj spec sheet), so RUL is expressed in months & cycles rather than as a   fraction of a rated cycle count. Plug in a rated count to convert.
- Projections are a **linear recent-rate extrapolation** (km, cycles, SoH fade) — a transparent baseline, not the   conditioned forecasting model. Build the Bajaj forecasting model (reuse `src/model.py`/`euler_model.py`) for   bands.
- These units are already <80% SoH (second-life candidates); EoL is set to 70% accordingly.